# BRITS imputation of the full dataset (downstream benchmark)

This notebook imputes the full dataset with the same two-tier rule as CAKGI, so the downstream hypoglycaemia benchmark (`fcn_hypo_benchmark.ipynb`) compares methods on equal terms:

| Gap length | Method |
|---|---|
| < 30 min | linear interpolation |
| 30 min – 8 h | BRITS |
| > 8 h | left missing |

BRITS runs bidirectionally over each patient's full 5-min grid, in overlapping chunks of `MAX_L` steps when the record is longer.

**Input:** `SDBII.csv` (`PtID`, `ts`, `GlucoseCGM`, `HR`).
**Output:** `SDBII_imputed_brits.csv`. `GlucoseCGM_imputed_final` and `HR_imputed_final` hold the observed or imputed values, and `imputation_method_*` marks each value as `real`, `linear_short`, `brits_long` or `missing`.

# Imports + Constants

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os, gc, warnings
warnings.filterwarnings('ignore')

BASE             = r"path/to/your/data_directory"     # <-- set this
ORIGINAL_FILE    = f"{BASE}/SDBII.csv"
MODEL_FILE       = f"{BASE}/SDBII_brits_model.pt"
FINAL_FILE       = f"{BASE}/SDBII_imputed_brits.csv"
CHECKPOINT_FILE  = f"{BASE}/SDBII_imputed_brits_ckpt.csv"

FREQ       = '5min'
CTX_WIN    = 120     # context window for diffusion methods
MAX_L      = 1500    # max window timesteps
T_STEPS    = 100
N_SAMPLES  = 50      # chains for diffusion methods (median aggregation)
SEQ_LEN    = 48
MASK_RATIO = 0.15
EPOCHS     = 75
BATCH_SIZE = 32
LR         = 1e-3
NF         = 2       # glucose + HR
HYPO_THR      = 70
MAX_GAP_MINS  = 480    # 8 hours — max imputable gap 

SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Method: brits | Device: {DEVICE}")


#  Load Data + Gap Classification

In [ ]:
df = pd.read_csv(ORIGINAL_FILE, sep=None, engine='python')
df['ts'] = pd.to_datetime(df['ts'], format='mixed', dayfirst=True)
df = df.sort_values(['PtID','ts']).reset_index(drop=True)
df['GlucoseCGM'] = df['GlucoseCGM'].replace(0, np.nan)
df['HR']         = df['HR'].replace(0, np.nan)


G_MEAN = df['GlucoseCGM'].mean(); G_STD = df['GlucoseCGM'].std()
H_MEAN = df['HR'].mean();          H_STD = df['HR'].std()
G_STD  = G_STD if G_STD > 0 else 1.
H_STD  = H_STD if H_STD > 0 else 1.

def classify_gaps(pat_df, col, freq='5min', thr=30):
    """
    Labels EVERY missing timestep in each gap:
      'small'   — gap < 30 min  -> linear interpolation
      'long'    — 30 min <= gap <= MAX_GAP_MINS (8hr) -> method imputation
      'toolong' — gap > 8hr -> left as missing 
    """
    full = pd.date_range(pat_df['ts'].min(), pat_df['ts'].max(), freq=freq)
    s    = pat_df.set_index('ts')[col].reindex(full)
    mis  = s.isna()
    lbl  = pd.Series(None, index=full, dtype=object)
    if mis.any():
        gid = (mis != mis.shift()).cumsum()[mis]
        for g, grp in s[mis].groupby(gid):
            dur_mins = len(grp) * 5
            if dur_mins < thr:
                gap_type = 'small'
            elif dur_mins <= MAX_GAP_MINS:
                gap_type = 'long'
            else:
                gap_type = 'toolong'   # > 8hr: skip imputation
            lbl.loc[grp.index] = gap_type
    result = pat_df['ts'].map(lbl[lbl.notna()].to_dict())
    result.index = pat_df.index
    return result

df['GlucoseCGM_gap'] = None; df['HR_gap'] = None
for pid, g in df.groupby('PtID'):
    df.loc[g.index,'GlucoseCGM_gap'] = classify_gaps(g,'GlucoseCGM').values
    df.loc[g.index,'HR_gap']         = classify_gaps(g,'HR').values

print(f"Loaded {len(df):,} rows | {df['PtID'].nunique()} patients")
print("Glucose gaps:", df['GlucoseCGM_gap'].value_counts(dropna=True).to_dict())
print("HR gaps:",      df['HR_gap'].value_counts(dropna=True).to_dict())


# BRITS 

In [ ]:
METHOD = 'brits'

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

NF = 2  # glucose + HR


class RITS(nn.Module):
    """
    Single direction of BRITS (the version used for the downstream benchmark):
    constant time step (delta_t = 1 for the regular 5-min grid), GRU cell.
    - Temporal decay on hidden state (gamma_h) and input (gamma_x)
    - History-based estimate from decayed hidden
    - Feature regression: estimate each signal from the OTHER signal
    - Beta-weighted combination of history and feature estimates
    - GRU update with decayed hidden state
    """
    def __init__(self, n_features=NF, hidden=64):
        super().__init__()
        self.n  = n_features
        self.h  = hidden

        # Temporal decay (for hidden state and input history)
        # For regular 5-min data delta_t=1
        self.W_gamma_h = nn.Linear(1, hidden, bias=True)     # hidden decay
        self.W_gamma_x = nn.Linear(1, n_features, bias=True) # input decay

        # History-based regression: x_hat from hidden state
        self.hist_reg  = nn.Linear(hidden, n_features)

        # Feature regression: estimate each feature from OTHER features
        # For 2-feature case: glucose from HR and vice versa
        self.feat_reg  = nn.Linear(n_features, n_features)

        # Beta combination weight: decides between feature and history estimate
        # Input: [gamma_h * h, delta_t (scalar per feature), mask]
        self.beta_net  = nn.Linear(hidden + 1 + n_features, n_features)

        # GRU cell: takes [filled_x, mask] as input
        self.gru       = nn.GRUCell(n_features * 2, hidden)

        # Output regression
        self.out_reg   = nn.Linear(hidden, n_features)

    def forward(self, x, mask):
        """
        x    : (B, L, NF) — signal, 0 at missing positions
        mask : (B, L, NF) — 1=observed, 0=missing
        Returns estimates : (B, L, NF), hidden_last : (B, H)
        """
        B, L, NF_ = x.shape   # NF_ avoids shadowing torch.nn.functional alias
        h   = torch.zeros(B, self.h, device=x.device)
        estimates = []
        relu = torch.nn.functional.relu   # local alias — avoids F being overwritten

        for t in range(L):
            # Delta_t = 1 for regular 5-min CGM data
            delta_t = torch.ones(B, 1, device=x.device)

            # Temporal decay 
            gamma_h = torch.exp(-relu(self.W_gamma_h(delta_t)))  # (B, hidden)
            gamma_x = torch.exp(-relu(self.W_gamma_x(delta_t)))  # (B, NF)

            # Decay hidden state
            h_hat = gamma_h * h

            # History-based estimate from decayed hidden state
            x_hist = self.hist_reg(h_hat)  # (B, NF)

            # Feature regression: estimate each feature from OTHER features
            # Uses observed values only (mask zeros out missing)
            x_obs_t = x[:, t, :] * mask[:, t, :]
            x_feat  = self.feat_reg(x_obs_t)          # (B, NF)

            # Beta-weighted combination 
            beta_in = torch.cat([h_hat, delta_t, mask[:, t, :]], dim=-1)
            beta    = torch.sigmoid(self.beta_net(beta_in))
            c_t     = beta * x_feat + (1 - beta) * x_hist

            # Fill observed positions with true values
            x_filled = mask[:, t, :] * x[:, t, :] + (1 - mask[:, t, :]) * c_t

            # GRU update with decayed hidden state
            gru_in = torch.cat([x_filled, mask[:, t, :]], dim=-1)
            h      = self.gru(gru_in, h_hat)

            estimates.append(c_t)

        return torch.stack(estimates, dim=1), h


class BRITSModel(nn.Module):
    """
    Full bidirectional BRITS
    """
    def __init__(self, n_features=NF, hidden=64):
        super().__init__()
        self.fwd = RITS(n_features, hidden)
        self.bwd = RITS(n_features, hidden)

    def forward(self, x, mask):
        """Returns imputed signal (B, L, F)"""
        # Forward
        est_f, _ = self.fwd(x, mask)
        # Backward (flip time)
        est_b, _ = self.bwd(
            torch.flip(x,    [1]),
            torch.flip(mask, [1])
        )
        est_b = torch.flip(est_b, [1])
        # Average forward and backward
        return (est_f + est_b) / 2

    def loss(self, x, mask):
        """
        x    : (B, L, F) FULL clean signal
        mask : (B, L, F) 1=conditioning/observed, 0=target
        """
        est_f, _ = self.fwd(x, mask)
        est_b, _ = self.bwd(
            torch.flip(x,    [1]),
            torch.flip(mask, [1])
        )
        est_b = torch.flip(est_b, [1])

        # Reconstruction loss at TARGET positions
        tgt = (1 - mask).bool()
        rec = torch.nn.functional.mse_loss(est_f[tgt], x[tgt]) + torch.nn.functional.mse_loss(est_b[tgt], x[tgt])

        # Consistency loss at MISSING positions 
        cons = torch.nn.functional.mse_loss(est_f[tgt], est_b[tgt].detach())

        return rec + 0.1 * cons


class CGMDatasetBRITS(Dataset):
    def __init__(self,df,seq=SEQ_LEN,mr=MASK_RATIO):
        self.s=[]
        for pid,p in df.groupby('PtID'):
            p=p.sort_values('ts').reset_index(drop=True)
            g=(p['GlucoseCGM'].values-G_MEAN)/G_STD
            h=(p['HR'].values-H_MEAN)/H_STD
            sig=np.stack([g,h],axis=1)
            for i in range(0,len(sig)-seq,seq//2):
                w=sig[i:i+seq]
                if np.isnan(w).any(): continue
                self.s.append(w.copy())
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        x=torch.tensor(self.s[i],dtype=torch.float32)
        m=torch.ones_like(x)
        nm=max(1,int(SEQ_LEN*MASK_RATIO))
        st=int(torch.randint(0,SEQ_LEN-nm+1,(1,)).item())
        m[st:st+nm]=0
        return x,m

model=BRITSModel().to(DEVICE)
print(f"BRITS: {sum(p.numel() for p in model.parameters()):,} params")
print("  Features: temporal decay + feature regression + beta combination")

if os.path.exists(MODEL_FILE):
    model.load_state_dict(torch.load(MODEL_FILE,map_location=DEVICE))
    print("Model loaded")
else:
    ds=CGMDatasetBRITS(df); ld=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True)
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS)
    model.train()
    for ep in range(EPOCHS):
        el=0
        for x,m in ld:
            opt.zero_grad(); l=model.loss(x.to(DEVICE),m.to(DEVICE))
            l.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); el+=l.item()
        sch.step()
        if (ep+1)%10==0: print(f"  Ep{ep+1}/{EPOCHS} loss={el/len(ld):.4f}")
    torch.save(model.state_dict(),MODEL_FILE)
    print(f"Model saved: {MODEL_FILE}")


# Short Gap Imputation (linear, < 30 min)

In [ ]:
def impute_short_gaps(df, freq=FREQ):
    """
    Linear interpolation for gaps < 30 min (< 6 timesteps = 'small').
    Modifies GlucoseCGM and HR columns directly.
    """
    df_out = df.copy()
    short_records = []   # track filled positions for label assignment

    for pid, pat in df_out.groupby('PtID'):
        pat_sorted = pat.sort_values('ts')
        for col, gap_col in [('GlucoseCGM','GlucoseCGM_gap'),('HR','HR_gap')]:
            # Find positions labelled 'small'
            small_mask = pat_sorted[gap_col] == 'small'
            if not small_mask.any(): continue

            # Interpolate the full series then keep only 'small' gap positions
            s  = pat_sorted.set_index('ts')[col]
            si = s.interpolate(method='linear', limit_area='inside')

            # Only fill where: (a) originally NaN, (b) gap_col == 'small'
            for idx in pat_sorted[small_mask].index:
                ts_val = pat_sorted.loc[idx, 'ts']
                if pd.isna(pat_sorted.loc[idx, col]):
                    filled_val = si.get(ts_val, np.nan)
                    if not pd.isna(filled_val):
                        df_out.loc[idx, col] = filled_val
                        short_records.append({
                            'PtID': pid, 'ts': ts_val, 'col': col,
                            'value': round(float(filled_val), 3)
                        })

    short_df = pd.DataFrame(short_records) if short_records else pd.DataFrame(
        columns=['PtID','ts','col','value'])
    n_g = (short_df['col']=='GlucoseCGM').sum() if len(short_df) else 0
    n_h = (short_df['col']=='HR').sum() if len(short_df) else 0
    print(f"Short gaps filled: Glucose={n_g:,} | HR={n_h:,} timesteps")
    return df_out.reset_index(drop=True), short_df

df_short, short_df = impute_short_gaps(df)


# Impute Long Gaps with BRITS

In [ ]:
from tqdm import tqdm
import pandas as pd

model.eval(); long_rows = []

for pid, pat in tqdm(df_short.groupby('PtID'), desc='BRITS'):
    pat = pat.sort_values('ts').reset_index(drop=True)
    
    # Reindex to full 5-min grid for this patient
    fidx = pd.date_range(pat['ts'].min(), pat['ts'].max(), freq=FREQ)
    gn   = pat.set_index('ts')['GlucoseCGM'].reindex(fidx).values
    hn   = pat.set_index('ts')['HR'].reindex(fidx).values
    gcls = pat.set_index('ts')['GlucoseCGM_gap'].reindex(fidx).values
    hcls = pat.set_index('ts')['HR_gap'].reindex(fidx).values
    
    orig_g_nan = np.isnan(gn)   # remember which were originally missing
    orig_h_nan = np.isnan(hn)
    
    # Normalise
    gn_n = (gn - G_MEAN) / G_STD
    hn_n = (hn - H_MEAN) / H_STD
    sig  = np.stack([gn_n, hn_n], axis=1)             # (L, 2)
    msk  = (~np.isnan(sig)).astype(np.float32)         # 1=observed
    sig  = np.nan_to_num(sig, nan=0.)
    
    L = len(fidx)
    if L > MAX_L:
        # Process in chunks if too long
        chunk = MAX_L - 48    # overlap of 48 steps
        step  = MAX_L - 96
        out   = np.zeros((L, NF), dtype=np.float32)
        cnt   = np.zeros(L, dtype=np.float32)
        for start in range(0, L, step):
            end = min(start + MAX_L, L)
            xs  = torch.tensor(sig[start:end], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            ms  = torch.tensor(msk[start:end], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                o = model(xs, ms).squeeze(0).cpu().numpy()
            out[start:end] += o
            cnt[start:end] += 1
        out /= cnt.reshape(-1, 1).clip(min=1)
    else:
        x   = torch.tensor(sig, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        m   = torch.tensor(msk, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = model(x, m).squeeze(0).cpu().numpy()  # (L, 2)
    
    # Record imputed values only for LONG gap positions 
    for j, ts_val in enumerate(fidx):
        # Glucose: record if originally NaN AND classified as long gap
        if orig_g_nan[j] and gcls[j] == 'long':   # 'toolong' excluded by classify_gaps
            long_rows.append({
                'PtID': pid, 'ts': ts_val, 'col': 'GlucoseCGM',
                'value': round(float(out[j, 0] * G_STD + G_MEAN), 3)
            })
        # HR: record if originally NaN AND classified as long gap
        if orig_h_nan[j] and hcls[j] == 'long':   # 'toolong' excluded by classify_gaps
            long_rows.append({
                'PtID': pid, 'ts': ts_val, 'col': 'HR',
                'value': round(float(out[j, 1] * H_STD + H_MEAN), 3)
            })

long_df = pd.DataFrame(long_rows)
print(f"BRITS long gaps imputed: {len(long_df):,} timesteps")
print(f"  Glucose: {(long_df['col']=='GlucoseCGM').sum():,}")
print(f"  HR:      {(long_df['col']=='HR').sum():,}")


#  Final Assembly + Labels + Save

In [ ]:
# Labels applied in order:
#   1. 'real'         — original observed value (never NaN)
#   2. 'linear_short' — linear interp filled (was NaN, gap < 30 min)
#   3. 'brits_long'— method filled (was NaN, gap >= 30 min)
#   4. 'missing'      — still NaN (includes 'toolong' gaps > 8hr)

df_final = df.copy()   

df_final['GlucoseCGM_imputed_final'] = df_final['GlucoseCGM'].copy()
df_final['HR_imputed_final']         = df_final['HR'].copy()

# Base labels from original data
df_final['imputation_method_glucose'] = np.where(
    df_final['GlucoseCGM'].isna(), 'missing', 'real')
df_final['imputation_method_hr'] = np.where(
    df_final['HR'].isna(), 'missing', 'real')

df_final['ts'] = pd.to_datetime(df_final['ts'])

# apply short gap imputations + label 'linear_short' 
if not short_df.empty:
    short_df['ts'] = pd.to_datetime(short_df['ts'])
    for col, imp_col, method_col in [
        ('GlucoseCGM','GlucoseCGM_imputed_final','imputation_method_glucose'),
        ('HR',        'HR_imputed_final',         'imputation_method_hr'),
    ]:
        s = short_df[short_df['col']==col][['PtID','ts','value']].copy()
        if s.empty: continue
        s = s.drop_duplicates(subset=['PtID','ts'])
        df_final = df_final.merge(
            s.rename(columns={'value': f'_s{col}'}),
            on=['PtID','ts'], how='left')
        # Fill where: short value exists AND original was missing
        sm = df_final[f'_s{col}'].notna() & df_final[imp_col].isna()
        df_final.loc[sm, imp_col]     = df_final.loc[sm, f'_s{col}']
        df_final.loc[sm, method_col] = 'linear_short'
        df_final.drop(columns=[f'_s{col}'], inplace=True)

#  apply long gap imputations + label 'brits_long'
if long_df is not None and len(long_df) > 0:
    long_df['ts'] = pd.to_datetime(long_df['ts'])
    for col, imp_col, method_col in [
        ('GlucoseCGM','GlucoseCGM_imputed_final','imputation_method_glucose'),
        ('HR',        'HR_imputed_final',         'imputation_method_hr'),
    ]:
        lg = long_df[long_df['col']==col][['PtID','ts','value']].copy()
        if lg.empty: continue
        lg = lg.drop_duplicates(subset=['PtID','ts'], keep='last')
        df_final = df_final.merge(
            lg.rename(columns={'value': f'_l{col}'}),
            on=['PtID','ts'], how='left')
        # Fill where: long value exists AND still missing after short-gap fill
        lm = df_final[f'_l{col}'].notna() & df_final[imp_col].isna()
        df_final.loc[lm, imp_col]     = df_final.loc[lm, f'_l{col}']
        df_final.loc[lm, method_col] = f'brits_long'
        df_final.drop(columns=[f'_l{col}'], inplace=True)

# ── Summary ───────────────────────────────────────────────────────
df_final.to_csv(FINAL_FILE, index=False)
print(f"\nSaved: {FINAL_FILE}")
print(f"Rows: {len(df_final):,}")
print(f"\nGlucose label distribution:")
print(df_final['imputation_method_glucose'].value_counts(dropna=False).to_string())
print(f"\nHR label distribution:")
print(df_final['imputation_method_hr'].value_counts(dropna=False).to_string())
g_nan = df_final['GlucoseCGM_imputed_final'].isna().sum()
h_nan = df_final['HR_imputed_final'].isna().sum()
print(f"\nRemaining NaN: Glucose={g_nan:,} | HR={h_nan:,}")
if g_nan > 0:
    print("  (NaN = gaps that could not be imputed due to insufficient context)")
